# Module chuẩn bị: Giải bài KKT từng bước — code

Notebook đi kèm site *Tối ưu hóa phi tuyến & lồi*. Chạy tuần tự từng ô (Colab có sẵn numpy, scipy, sympy, cvxpy). Tự giải tay trước rồi đối chiếu kết quả.

## Bài A: cvxpy, kiểm DCP và nhân tử (đoạn code trong file của bạn)

Không cần qcp=True vì bài đã lồi (DCP). Hai bán kính: 6 (cực tiểu tự do khả thi) và 4 (một ràng buộc chặt).

In [1]:
import cvxpy as cp
for R2 in (36, 16):
    x, y = cp.Variable(), cp.Variable()
    cons = [x**2 + y**2 <= R2, x >= 0, y >= 0]
    prob = cp.Problem(cp.Minimize((x - 3)**2 + (y - 4)**2 - 2), cons)
    prob.solve()                       # không cần qcp=True
    print("R^2 =", R2, "| DCP:", prob.is_dcp(), "| x, y =", round(float(x.value), 4), round(float(y.value), 4), "| f* =", round(prob.value, 4))
    print("   lambda =", [round(float(c.dual_value), 4) for c in cons])

R^2 = 36 | DCP: True | x, y = 3.0 4.0 | f* = -2.0
   lambda = [0.0, 0.0, 0.0]
R^2 = 16 | DCP: True | x, y = 2.4 3.2 | f* = -1.0
   lambda = [0.25, 0.0, 0.0]


## Bài B (slide Ví dụ 1): giải KKT bằng sympy

Giải theo λ như trong bài giảng, in ca nào là KKT.

In [2]:
import sympy as sp
x, y, lam, mu = sp.symbols("x y lam mu", real=True)
f = (x - 1)**2 + y - 2; g = x + y - 2; h = x - y + 1
L = f + lam*g + mu*h
stat = [sp.diff(L, x), sp.diff(L, y), h]
for name, extra, sub in (("g không chặt (lam=0)", [], {lam: 0}), ("g chặt", [g], {})):
    vs = [v for v in (x, y, lam, mu) if v not in sub]
    for s in sp.solve([e.subs(sub) for e in stat] + extra, vs, dict=True):
        s = {**sub, **s}
        ok = g.subs(s) <= 0 and s[lam] >= 0
        print(name, {k: str(v) for k, v in s.items()}, "-> KKT" if ok else "-> loại")
print("f* =", f.subs({x: sp.Rational(1, 2), y: sp.Rational(3, 2)}))

g không chặt (lam=0) {lam: '0', mu: '1', x: '1/2', y: '3/2'} -> KKT
g chặt {lam: '0', mu: '1', x: '1/2', y: '3/2'} -> KKT
f* = -1/4


## Bài B bằng cvxpy: λ và μ

Ràng buộc đẳng thức cho dual_value = μ (ở đây = 1), ràng buộc BĐT cho λ (= 0 dù chặt).

In [3]:
import cvxpy as cp
x, y = cp.Variable(), cp.Variable()
cons = [x + y - 2 <= 0, x - y + 1 == 0]
prob = cp.Problem(cp.Minimize((x - 1)**2 + y - 2), cons); prob.solve()
print("x, y =", round(float(x.value), 4), round(float(y.value), 4), " f* =", round(prob.value, 4))
print("lambda (BĐT) =", round(float(cons[0].dual_value), 4), " mu (đẳng thức) =", round(float(cons[1].dual_value), 4))
print("g tại nghiệm =", round(float(x.value + y.value - 2), 6), "(chặt nhưng lambda = 0)")

x, y = 0.5 1.5  f* = -0.25
lambda (BĐT) = 0.0  mu (đẳng thức) = 1.0
g tại nghiệm = 0.0 (chặt nhưng lambda = 0)


## Bài C: điểm KKT không phải cực tiểu

Thử nhiều điểm đầu với scipy; so giá trị tại x=0 và x=±1.

In [4]:
import numpy as np
from scipy.optimize import minimize
f = lambda v: -v[0]**2
cons = [{"type": "ineq", "fun": lambda v: 1 - v[0]**2}]
for x0 in (-0.9, -0.2, 0.2, 0.9):
    r = minimize(f, [x0], constraints=cons, method="SLSQP")
    print("x0 =", x0, "->", round(float(r.x[0]), 4), " f =", round(float(r.fun), 4))
print("f(0) =", f([0.0]), " (điểm KKT nhưng là cực đại), f(±1) =", f([1.0]))

x0 = -0.9 -> -1.0  f = -1.0
x0 = -0.2 -> -1.0  f = -1.0
x0 = 0.2 -> 1.0  f = -1.0
x0 = 0.9 -> 1.0  f = -1.0
f(0) = -0.0  (điểm KKT nhưng là cực đại), f(±1) = -1.0


## Bài code · Chiếu lên hình tròn (lối tắt Bài A)

Đầu vào: tâm $c=(a,b)$, bán kính $R$. Đầu ra: nghiệm của $\min\lVert x-c\rVert^2$ s.t. $\lVert x\rVert\le R$ và nhân tử $\lambda$.

In [5]:
import numpy as np
def project_ball(c, R):
    c = np.array(c, float); n = np.linalg.norm(c)
    if n <= R: return c, 0.0                    # cực tiểu tự do khả thi: mọi lambda = 0
    return c * R / n, n / R - 1                 # ràng buộc chặt: hình chiếu, lambda = n/R - 1
for c, R in (((3, 4), 6), ((3, 4), 4), ((6, 8), 5)):
    x, lam = project_ball(c, R); print(c, R, "->", x.round(4), "lambda =", round(lam, 4))

(3, 4) 6 -> [3. 4.] lambda = 0.0
(3, 4) 4 -> [2.4 3.2] lambda = 0.25
(6, 8) 5 -> [3. 4.] lambda = 1.0


## Bài code · Kiểm điểm KKT của Bài B

Đầu vào: điểm $(x,y)$. Đầu ra: $g$, $h$, nghiệm $(\lambda,\mu)$ của phương trình dừng và kết luận KKT.

In [6]:
import numpy as np
def check_B(x, y, tol=1e-9):
    gf = np.array([2*(x - 1), 1.0]); gg = np.array([1.0, 1.0]); gh = np.array([1.0, -1.0])
    g, h = x + y - 2, x - y + 1
    sol = np.linalg.solve(np.column_stack([gg, gh]), -gf)      # gf + lam*gg + mu*gh = 0
    lam, mu = sol
    ok = abs(h) < tol and g <= tol and lam >= -tol and abs(lam * g) < tol
    return g, h, lam, mu, ok
print(check_B(0.5, 1.5))
print(check_B(0.0, 1.0))

(0.0, 0.0, np.float64(0.0), np.float64(1.0), np.True_)
(-1.0, 0.0, np.float64(0.5), np.float64(1.5), np.False_)
